## Parameter Tuning (Chunked Data)

**IMPORTANT**: This notebook tunes parameters on **semantically chunked data** to match the production pipeline. Documents are split into chunks only if they exceed 7000 tokens, so most documents remain intact. Expect ~165-200 data points (original documents + some chunks from very long documents).

This notebook serves as a tool to find optimal parameters for the BERTopic model. We test for the following metrics (influenced by Murugaraj et al., 2025 - "Mining the Past"):

### **1. Coherence (C_V Score)**
- **Measures**: How semantically related the top words in each topic are
- **Calculation**: Uses word co-occurrence statistics + word embeddings to check if topic words "belong together"
- **Goal**: **HIGH** (> 0.5 good, > 0.7 excellent)
- **Interpretation**:
  - High (0.7+): Topic words strongly co-occur in documents (e.g., "computer, program, code")
  - Low (<0.4): Random word collections that don't make semantic sense
- **Example**: Topic with words ["machine", "learning", "neural", "network"] = high coherence

### **2. Diversity**
- **Measures**: How unique/different topics are from each other
- **Calculation**: `unique_words_across_all_topics / total_words_in_all_topics`
- **Goal**: **HIGH** (> 0.7 good, > 0.85 excellent)
- **Interpretation**:
  - High (0.85+): Each topic has unique vocabulary, minimal overlap
  - Low (<0.6): Topics share too many words, aren't distinct
- **Trade-off**: Sometimes conflicts with coherence (very specific topics = high diversity but may have lower coherence)

### **3. Silhouette Score**
- **Measures**: How well-separated clusters are in embedding space
- **Calculation**: `(distance_to_nearest_other_cluster - distance_within_own_cluster) / max(both)`
- **Goal**: **HIGH** (> 0.3 good, > 0.5 excellent)
- **Interpretation**:
  - High (0.5+): Documents clearly belong to their assigned topic
  - Low (<0.2): Fuzzy boundaries, documents could fit multiple topics
  - Negative: Document is closer to another cluster than its own

### **4. Number of Topics**
- **Measures**: How many distinct topics were identified
- **Goal**: **10-15 topics** (target similar to unchunked due to high chunk threshold)
- **Rule of thumb**: ~1 topic per 10-15 data points
- **Interpretation**:
  - Too many (>20): Over-fragmentation, topics too granular
  - Too few (<8): Under-segmentation, topics too broad

### **5. Outlier Percentage**
- **Measures**: % of documents assigned to topic -1 (no clear topic)
- **Goal**: **LOW** (< 15% good, < 10% excellent)
- **Interpretation**:
  - Low (<10%): Most documents successfully clustered
  - High (>25%): Many documents don't fit any topic (too restrictive parameters)

### **6. Composite Score**
- **Weighted combination**:
  - 40% Coherence
  - 25% Diversity
  - 15% Outlier penalty
  - 10% Silhouette
  - 10% Topic count
Note - this is useful to judge parameter combinations at a glance but individual metrics will give more info
---

## Setup

In [1]:
# Environment Setup
import os
import zipfile
from pathlib import Path

# Detect environment - CHECK KAGGLE FIRST!
IN_KAGGLE = os.path.exists('/kaggle')

if not IN_KAGGLE:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
else:
    IN_COLAB = False

if IN_COLAB:
    # === COLAB SETUP ===
    # Mount Google Drive
    drive.mount('/content/drive')

    # Unzip project folder (only if not already extracted)
    zip_path = '/content/drive/MyDrive/Final_Year_Project_FYP.zip'
    extract_to = '/content/drive/MyDrive/'
    project_folder = Path(extract_to) / 'Final Year Project FYP'

    if project_folder.exists():
        print(f"Project folder already exists: {project_folder}")
        print("Skipping extraction.")
    else:
        print(f"Unzipping {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Unzipped to: {project_folder}")

    # Install dependencies
    print("\nInstalling dependencies...")
    os.system('pip install -q sentence-transformers bertopic hdbscan umap-learn plotly gensim')

    # Check GPU availability
    import torch
    print("\nChecking GPU...")
    if torch.cuda.is_available():
        print(f"GPU AVAILABLE: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"Embeddings will be 10-50x faster!")
    else:
        print("No GPU detected.")

    print("\nColab setup complete.")
    
elif IN_KAGGLE:
    # === KAGGLE SETUP ===
    # NOTE: Upload your entire project folder as a Kaggle dataset first
    # Dataset should be named something like 'final-year-project-fyp'
    
    # Update this to match your Kaggle dataset name
    dataset_name = 'final-year-project'  # CHANGE THIS to your actual dataset name
    dataset_path = Path(f'/kaggle/input/{dataset_name}')
    project_folder = dataset_path / 'Final Year Project FYP'
    
    if not project_folder.exists():
        print(f"ERROR: Project folder not found at {project_folder}")
        print(f"Please upload your project as a Kaggle dataset named '{dataset_name}'")
        print(f"Available datasets: {list(Path('/kaggle/input').glob('*'))}")
    else:
        print(f"✓ Project folder found: {project_folder}")
    
    # Install dependencies
    print("\nInstalling dependencies...")
    os.system('pip install -q sentence-transformers bertopic hdbscan umap-learn plotly gensim')
    
    # Check GPU
    import torch
    print("\nChecking GPU...")
    if torch.cuda.is_available():
        print(f"GPU AVAILABLE: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"Embeddings will be 10-50x faster!")
    else:
        print("No GPU detected.")
    
    print("\nKaggle setup complete.")

else:
    print("Running locally - no setup needed")

✓ Project folder found: /kaggle/input/final-year-project/Final Year Project FYP

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.0 MB/s eta 0:00:00

Checking GPU...
GPU AVAILABLE: Tesla T4
Memory: 15.8 GB
Embeddings will be 10-50x faster!

Kaggle setup complete.


In [2]:
import sys
from pathlib import Path

# Setup paths based on environment
if IN_COLAB:
    project_root = Path('/content/drive/MyDrive/Final Year Project FYP')
elif IN_KAGGLE:
    project_root = project_folder
else:
    project_root = Path.cwd().parent.parent.parent

src_path = project_root / 'final_pipeline' / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import logging
import warnings
import numpy as np
import pandas as pd
from datetime import datetime

# Import our modules
from parameter_tuning.helpers import ParameterTuner, generate_embeddings
from parameter_tuning.visualizations import ParameterVisualizer
from topic_modelling.config import load_config, save_config, TopicmodellingConfig
from topic_modelling.utils.data_utils import load_processed_data

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

warnings.filterwarnings('ignore')

print("Setup complete")
print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print(f"Python path: {sys.path[0]}")


2026-01-01 17:34:06.012594: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767288846.211961      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767288846.271219      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767288846.733993      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767288846.734031      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767288846.734034      55 computation_placer.cc:177] computation placer alr

Setup complete
Project root: /kaggle/input/final-year-project/Final Year Project FYP
Source path: /kaggle/input/final-year-project/Final Year Project FYP/final_pipeline/src
Python path: /kaggle/input/final-year-project/Final Year Project FYP/final_pipeline/src


## Configuration

In [3]:
# Paths
DATA_ROOT = project_root / 'data'  # Read from input
CONFIG_PATH = src_path / 'parameter_tuning' / 'configs' / 'baseline_config.json'

# Output directory - writable location
if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/tuning_results')
else:
    OUTPUT_DIR = src_path / 'parameter_tuning' / 'results'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Settings
ERA = 2  # Change to 1 for Era1

CACHE_EMBEDDINGS = True  # Set True to save embeddings for reuse

print(f"Data root: {DATA_ROOT}")
print(f"Config path: {CONFIG_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Testing Era: {ERA}")


Data root: /kaggle/input/final-year-project/Final Year Project FYP/data
Config path: /kaggle/input/final-year-project/Final Year Project FYP/final_pipeline/src/parameter_tuning/configs/baseline_config.json
Output directory: /kaggle/working/tuning_results
Testing Era: 2


## 1. Load Data and Apply Semantic Chunking

Loading documents and chunking them to match production pipeline.

In [4]:
# Load processed data
logger.info(f"Loading Era{ERA} data...")
documents, metadata = load_processed_data(DATA_ROOT, ERA)

print(f"\nLoaded {len(documents)} documents")
print(f"\nSample document:")
print(documents[0][:300] + "...")


Loaded 181 documents

Sample document:
publication date title unleashing power enduser programmable current issue past issues topics july volume issue pdf unleashing power enduser programmable creating aifirst program synthesis framework erik meijer demonstration accomplished contemporary llms large language model paper outline highlevel...


In [5]:
# Apply semantic chunking to match production pipeline
from topic_modelling.utils.chunker import SemanticChunker

chunker = SemanticChunker(
    max_tokens=5000,
    overlap_tokens=200,
    chunk_threshold=7000
)

logger.info(f"Applying semantic chunking...")
documents, metadata = chunker.chunk_documents(documents, metadata)

print(f"\nAfter chunking: {len(documents)} chunks")
print(f"\nSample chunk:")
print(documents[0][:300] + "...")


After chunking: 198 chunks

Sample chunk:
publication date title unleashing power enduser programmable current issue past issues topics july volume issue pdf unleashing power enduser programmable creating aifirst program synthesis framework erik meijer demonstration accomplished contemporary llms large language model paper outline highlevel...


## 2. Generate or Load Embeddings

In [6]:
# Load config and setup embedding cache path
config = load_config(CONFIG_PATH)
embedding_model_name = config.embedding_model

print(f"Embedding model: {embedding_model_name}")

# Setup cache path
cache_model_slug = embedding_model_name.replace('/', '_').replace('-', '_')
EMBEDDING_CACHE_PATH = OUTPUT_DIR / f'era{ERA}_embeddings_{cache_model_slug}.npy'

print(f"Cache path: {EMBEDDING_CACHE_PATH}")
print(f"Cache exists: {EMBEDDING_CACHE_PATH.exists()}")

Embedding model: Alibaba-NLP/gte-base-en-v1.5
Cache path: /kaggle/working/tuning_results/era2_embeddings_Alibaba_NLP_gte_base_en_v1.5.npy
Cache exists: False


In [7]:
# Load embedding model (separate cell to isolate timing to figure out why it takes so long ahhhhhh)
from sentence_transformers import SentenceTransformer
import time

if not EMBEDDING_CACHE_PATH.exists():
    print(f"Loading model: {embedding_model_name}")

    load_start = time.time()
    embedding_model = SentenceTransformer(embedding_model_name, trust_remote_code=True)
    load_time = time.time() - load_start

    # Check for GPU and move model if available
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        embedding_model = embedding_model.to(device)
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        print(f"Using CPU \n")

    print(f"Model loaded in {load_time:.1f}s")
    print(f"Device: {device}")
    print(f"Max sequence length: {embedding_model.max_seq_length}")
    print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

    # Check document lengths
    print(f"\n Document length analysis:")
    doc_lengths = [len(doc.split()) for doc in documents]
    print(f"Min length: {min(doc_lengths)} tokens")
    print(f"Max length: {max(doc_lengths)} tokens")
    print(f"Avg length: {sum(doc_lengths)/len(doc_lengths):.0f} tokens")

    if max(doc_lengths) > embedding_model.max_seq_length:
        print(f"\n{sum(1 for l in doc_lengths if l > embedding_model.max_seq_length)} docs exceed {embedding_model.max_seq_length} tokens (will be truncated)")

    # Test single-document encoding speed
    print("\Testing encoding speed...")
    test_start = time.time()
    _ = embedding_model.encode([documents[0]], show_progress_bar=False)
    test_time = time.time() - test_start
    print(f"   Single doc: {test_time:.2f}s")

    # Estimate total time
    estimated_total = test_time * len(documents)
    print(f"\Estimated total time: {estimated_total/60:.1f} minutes for {len(documents)} docs")
else:
    print("Embeddings cached - model loading not needed")

Loading model: Alibaba-NLP/gte-base-en-v1.5


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration.py: 0.00B [00:00, ?B/s]

... (6 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Using GPU: Tesla T4
Model loaded in 4.1s
Device: cuda
Max sequence length: 8192
Embedding dimension: 768

 Document length analysis:
Min length: 55 tokens
Max length: 4750 tokens
Avg length: 1544 tokens
\Testing encoding speed...
   Single doc: 0.63s
\Estimated total time: 2.1 minutes for 198 docs


In [8]:
# Generate or load embeddings
import time
import sys

if EMBEDDING_CACHE_PATH.exists():
    print(f"Loading cached embeddings...")
    embeddings = np.load(EMBEDDING_CACHE_PATH)
    print(f"Loaded {embeddings.shape[0]} embeddings (dim={embeddings.shape[1]})")
else:
    print(f"Generating embeddings for {len(documents)} documents...")

    batch_size = 8
    all_embeddings = []
    total_batches = (len(documents) + batch_size - 1) // batch_size

    print(f"Processing {total_batches} batches (batch_size={batch_size})...")
    sys.stdout.flush()
    start_time = time.time()

    for i in range(0, len(documents), batch_size):
        batch_num = (i // batch_size) + 1
        batch = documents[i:i + batch_size]

        batch_start = time.time()
        batch_embeddings = embedding_model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
        batch_time = time.time() - batch_start

        all_embeddings.append(batch_embeddings)

        # Clear GPU cache to prevent memory fragmentation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        docs_done = min(i + batch_size, len(documents))
        pct = (docs_done / len(documents)) * 100

        print(f"Batch {batch_num}/{total_batches} ({docs_done}/{len(documents)} docs, {pct:.1f}%) - {batch_time:.1f}s")
        sys.stdout.flush()

    embeddings = np.vstack(all_embeddings)
    total_time = time.time() - start_time
    print(f"\Complete! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")

    # Cache the embeddings
    np.save(EMBEDDING_CACHE_PATH, embeddings)
    print(f"Cached {embeddings.shape[0]} embeddings (dim={embeddings.shape[1]})")

Generating embeddings for 198 documents...
Processing 25 batches (batch_size=8)...
Batch 1/25 (8/198 docs, 4.0%) - 3.7s
Batch 2/25 (16/198 docs, 8.1%) - 0.8s
Batch 3/25 (24/198 docs, 12.1%) - 0.7s
Batch 4/25 (32/198 docs, 16.2%) - 0.9s
Batch 5/25 (40/198 docs, 20.2%) - 0.7s
Batch 6/25 (48/198 docs, 24.2%) - 0.8s
Batch 7/25 (56/198 docs, 28.3%) - 2.1s
Batch 8/25 (64/198 docs, 32.3%) - 0.8s
Batch 9/25 (72/198 docs, 36.4%) - 1.0s
Batch 10/25 (80/198 docs, 40.4%) - 0.8s
Batch 11/25 (88/198 docs, 44.4%) - 4.6s
Batch 12/25 (96/198 docs, 48.5%) - 11.2s
Batch 13/25 (104/198 docs, 52.5%) - 11.3s
Batch 14/25 (112/198 docs, 56.6%) - 11.5s
Batch 15/25 (120/198 docs, 60.6%) - 11.8s
Batch 16/25 (128/198 docs, 64.6%) - 7.5s
Batch 17/25 (136/198 docs, 68.7%) - 12.2s
Batch 18/25 (144/198 docs, 72.7%) - 3.8s
Batch 19/25 (152/198 docs, 76.8%) - 3.7s
Batch 20/25 (160/198 docs, 80.8%) - 0.8s
Batch 21/25 (168/198 docs, 84.8%) - 2.2s
Batch 22/25 (176/198 docs, 88.9%) - 5.5s
Batch 23/25 (184/198 docs, 92.9%) 

## 3. Baseline Test

Run with your current configuration to establish a baseline.

In [9]:
from topic_modelling.core.topic_modeler import TopicModeler

# Initialize tuner
tuner = ParameterTuner(config)
visualizer = ParameterVisualizer(OUTPUT_DIR)

# Run baseline
logger.info("Running baseline configuration...")
modeler = TopicModeler(config)
modeler.create_model()
# Use pre-computed embeddings to avoid re-embedding and OOM errors
baseline_topics, baseline_probs = modeler.fit_transform(documents, embeddings=embeddings)

# Evaluate
baseline_metrics = tuner.evaluate_model(
    modeler.model,
    documents,
    embeddings,
    baseline_topics,
    baseline_probs
)
baseline_metrics['composite_score'] = tuner.calculate_composite_score(baseline_metrics)

# Display results
print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
for key, value in baseline_metrics.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.3f}")
    else:
        print(f"{key:25s}: {value}")
print("="*60)


2026-01-01 17:36:31,212 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 17:36:37,719 - BERTopic - Dimensionality - Completed ✓
2026-01-01 17:36:37,721 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 17:36:37,754 - BERTopic - Cluster - Completed ✓
2026-01-01 17:36:37,761 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 17:36:38,774 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


BASELINE RESULTS
coherence                : 0.543
diversity                : 0.724
n_topics                 : 33
outlier_pct              : 0.247
avg_docs_per_topic       : 4.515
silhouette               : 0.13955460488796234
min_topic_size           : 2
max_topic_size           : 11
median_topic_size        : 4
composite_score          : 0.599


In [10]:
# Visualize baseline
baseline_umap = visualizer.plot_umap_2d(
    embeddings,
    baseline_topics,
    documents,
    metadata,
    title=f"Baseline Configuration - Era{ERA} Topics",
    save_name=f"baseline_era{ERA}_umap.html"
)

print(f"UMAP plot saved to: {OUTPUT_DIR / f'baseline_era{ERA}_umap.html'}")

baseline_dist = visualizer.plot_topic_distribution(
    baseline_topics,
    title="Baseline - Documents per Topic",
    save_name=f"baseline_era{ERA}_distribution.html"
)
print(f"Distribution plot saved to: {OUTPUT_DIR / f'baseline_era{ERA}_distribution.html'}")

UMAP plot saved to: /kaggle/working/tuning_results/baseline_era2_umap.html
Distribution plot saved to: /kaggle/working/tuning_results/baseline_era2_distribution.html


## 4. Sensitivity Analysis

Test each parameter individually to understand its impact.

### 4.1 Test min_cluster_size (MOST IMPORTANT)

**Note**: Working with **~165-200 data points** after semantic chunking (7000 token threshold means most docs stay intact). Targeting **10-15 topics**.

In [11]:
# Test min_cluster_size: controls topic granularity
# With chunks: more data points, so test slightly larger values
# Smaller = more granular topics; Larger = broader topics
min_cluster_sizes = [3, 4, 5, 6, 7, 8, 9, 10]

logger.info("Testing min_cluster_size sensitivity...")
mcs_results = tuner.sensitivity_test(
    documents,
    embeddings,
    'hdbscan.min_cluster_size',
    min_cluster_sizes
)

# Display results
print("\nmin_cluster_size Results:")
print(mcs_results[['value', 'n_topics', 'coherence', 'diversity', 'outlier_pct', 'composite_score']].to_string(index=False))

2026-01-01 17:36:47,858 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 17:36:48,319 - BERTopic - Dimensionality - Completed ✓
2026-01-01 17:36:48,320 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 17:36:48,341 - BERTopic - Cluster - Completed ✓
2026-01-01 17:36:48,343 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 17:36:49,547 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


min_cluster_size Results:
 value  n_topics  coherence  diversity  outlier_pct  composite_score
     3        27   0.541016   0.714815     0.242424         0.602796
     4        17   0.561296   0.735294     0.303030         0.627782
     5        13   0.545463   0.715385     0.363636         0.624287
     6        11   0.547050   0.709091     0.414141         0.629671
     7         7   0.526987   0.742857     0.282828         0.633537
     8         6   0.530215   0.783333     0.318182         0.634017
     9         6   0.530215   0.783333     0.318182         0.634017
    10         6   0.530215   0.783333     0.318182         0.634017


In [12]:
# Visualize min_cluster_size sensitivity
mcs_plot = visualizer.plot_metric_comparison(
    mcs_results,
    'value',
    metrics=['coherence', 'diversity', 'n_topics', 'outlier_pct'],
    title="min_cluster_size Sensitivity",
    save_name=f"sensitivity_min_cluster_size_era{ERA}.html"
)
print(f"Plot saved to: {OUTPUT_DIR / f'sensitivity_min_cluster_size_era{ERA}.html'}")

Plot saved to: /kaggle/working/tuning_results/sensitivity_min_cluster_size_era2.html


### 4.2 Test n_neighbors

In [13]:
# Test n_neighbors: controls local vs global structure
# With chunks: test wider range for better global structure
# Smaller = more local clusters; Larger = more global structure
n_neighbors_values = [10, 15, 20, 25, 30, 40, 50]

logger.info("Testing n_neighbors sensitivity...")
nn_results = tuner.sensitivity_test(
    documents,
    embeddings,
    'umap.n_neighbors',
    n_neighbors_values
)

print("\nn_neighbors Results:")
print(nn_results[['value', 'n_topics', 'coherence', 'diversity', 'outlier_pct', 'composite_score']].to_string(index=False))

2026-01-01 17:37:52,777 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 17:37:53,090 - BERTopic - Dimensionality - Completed ✓
2026-01-01 17:37:53,091 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 17:37:53,112 - BERTopic - Cluster - Completed ✓
2026-01-01 17:37:53,115 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 17:37:54,137 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


n_neighbors Results:
 value  n_topics  coherence  diversity  outlier_pct  composite_score
    10        28   0.563218   0.707143     0.161616         0.619159
    15        27   0.544333   0.718519     0.171717         0.614788
    20        29   0.542150   0.696552     0.196970         0.602199
    25        29   0.533588   0.713793     0.176768         0.605962
    30        33   0.543337   0.724242     0.247475         0.598555
    40        29   0.558369   0.696552     0.227273         0.604639
    50        31   0.542108   0.683871     0.191919         0.596568


In [14]:
# Visualize n_neighbors sensitivity
nn_plot = visualizer.plot_metric_comparison(
    nn_results,
    'value',
    metrics=['coherence', 'diversity', 'n_topics', 'outlier_pct', 'composite_score'],
    title="n_neighbors Sensitivity",
    save_name=f"sensitivity_n_neighbors_era{ERA}.html"
)
print(f"Plot saved to: {OUTPUT_DIR / f'sensitivity_n_neighbors_era{ERA}.html'}")

Plot saved to: /kaggle/working/tuning_results/sensitivity_n_neighbors_era2.html


### 4.3 Test n_components

In [15]:
# Test n_components: dimensionality after UMAP reduction
# With chunks: may need higher dimensionality to capture complexity
# Fewer = simpler; More = captures complexity
n_components_values = [4, 5, 6, 7, 8, 9, 10]

logger.info("Testing n_components sensitivity...")
nc_results = tuner.sensitivity_test(
    documents,
    embeddings,
    'umap.n_components',
    n_components_values
)

print("\nn_components Results:")
print(nc_results[['value', 'n_topics', 'coherence', 'diversity', 'outlier_pct', 'composite_score']].to_string(index=False))

2026-01-01 17:38:58,994 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 17:38:59,433 - BERTopic - Dimensionality - Completed ✓
2026-01-01 17:38:59,433 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 17:38:59,458 - BERTopic - Cluster - Completed ✓
2026-01-01 17:38:59,461 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 17:39:00,536 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


n_components Results:
 value  n_topics  coherence  diversity  outlier_pct  composite_score
     4        35   0.565576   0.677143     0.176768         0.603933
     5        33   0.543337   0.724242     0.247475         0.598555
     6        28   0.554453   0.700000     0.212121         0.606423
     7        28   0.552489   0.728571     0.217172         0.612642
     8        28   0.555811   0.739286     0.202020         0.619468
     9        31   0.566206   0.732258     0.212121         0.616456
    10        34   0.553506   0.694118     0.191919         0.602639


In [16]:
# Visualize n_components sensitivity
nc_plot = visualizer.plot_metric_comparison(
    nc_results,
    'value',
    metrics=['coherence', 'diversity', 'n_topics', 'outlier_pct'],
    title="n_components Sensitivity",
    save_name=f"sensitivity_n_components_era{ERA}.html"
)
print(f"Plot saved to: {OUTPUT_DIR / f'sensitivity_n_components_era{ERA}.html'}")

Plot saved to: /kaggle/working/tuning_results/sensitivity_n_components_era2.html


### 4.4 Test min_samples

In [17]:
# Test min_samples: cluster density requirement
# With chunks: test slightly wider range
# Lower = more lenient clustering; Higher = stricter
min_samples_values = [2, 3, 4, 5, 6, 7, 8, 10]

logger.info("Testing min_samples sensitivity...")
ms_results = tuner.sensitivity_test(
    documents,
    embeddings,
    'hdbscan.min_samples',
    min_samples_values
)

print("\nmin_samples Results:")
print(ms_results[['value', 'n_topics', 'coherence', 'diversity', 'outlier_pct', 'composite_score']].to_string(index=False))

2026-01-01 17:40:06,678 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 17:40:07,143 - BERTopic - Dimensionality - Completed ✓
2026-01-01 17:40:07,144 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 17:40:07,166 - BERTopic - Cluster - Completed ✓
2026-01-01 17:40:07,169 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 17:40:08,198 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


min_samples Results:
 value  n_topics  coherence  diversity  outlier_pct  composite_score
     2        33   0.543337   0.724242     0.247475         0.598555
     3        27   0.582520   0.740741     0.313131         0.614686
     4        15   0.543869   0.733333     0.323232         0.623832
     5        10   0.539999   0.740000     0.368687         0.650422
     6        11   0.568298   0.763636     0.434343         0.649011
     7        11   0.553164   0.754545     0.530303         0.625624
     8         9   0.538367   0.788889     0.540404         0.626783
    10         2   0.472937   0.900000     0.318182         0.623936


In [18]:
# Visualize min_samples sensitivity
ms_plot = visualizer.plot_metric_comparison(
    ms_results,
    'value',
    metrics=['coherence', 'diversity', 'n_topics', 'outlier_pct'],
    title="min_samples Sensitivity",
    save_name=f"sensitivity_min_samples_era{ERA}.html"
)
print(f"Plot saved to: {OUTPUT_DIR / f'sensitivity_min_samples_era{ERA}.html'}")

Plot saved to: /kaggle/working/tuning_results/sensitivity_min_samples_era2.html


### 4.5 Combine Sensitivity Results

In [19]:
# Combine all sensitivity results
all_sensitivity = pd.concat([
    mcs_results.assign(parameter='min_cluster_size'),
    nn_results.assign(parameter='n_neighbors'),
    nc_results.assign(parameter='n_components'),
    ms_results.assign(parameter='min_samples')
])

# Save
sensitivity_path = OUTPUT_DIR / f'sensitivity_analysis_era{ERA}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
all_sensitivity.to_csv(sensitivity_path, index=False)
print(f"\n✓ Saved sensitivity results to {sensitivity_path}")

# Show top performers per parameter
print("\nBest value per parameter (by composite score):")
print("=" * 60)
for param in ['min_cluster_size', 'n_neighbors', 'n_components', 'min_samples']:
    param_results = all_sensitivity[all_sensitivity['parameter'] == param]
    best = param_results.nlargest(1, 'composite_score').iloc[0]
    print(f"{param:20s}: {best['value']:6} (score={best['composite_score']:.3f}, coherence={best['coherence']:.3f})")


✓ Saved sensitivity results to /kaggle/working/tuning_results/sensitivity_analysis_era2_20260101_174110.csv

Best value per parameter (by composite score):
min_cluster_size    :      8 (score=0.634, coherence=0.530)
n_neighbors         :     10 (score=0.619, coherence=0.563)
n_components        :      8 (score=0.619, coherence=0.556)
min_samples         :      5 (score=0.650, coherence=0.540)


## 5. Grid Search

Test combinations of the best-performing values from sensitivity analysis.

In [44]:
# Define parameter grid based on sensitivity results
# IMPORTANT: Update these ranges after running sensitivity analysis above
# Use the top-performing values from each parameter test

param_grid = {
    'hdbscan.min_cluster_size': [3, 4, 7],  # Update after sensitivity test
    
    'umap.n_neighbors': [10, 15],  # Update after sensitivity test
    
    'umap.n_components': [4, 6, 8, 10],  # Update after sensitivity test
    
    'hdbscan.min_samples': [3, 4, 5]  # Update after sensitivity test
}

# Calculate total combinations
total_combos = 1
for values in param_grid.values():
    total_combos *= len(values)

print(f"Grid search will test {total_combos} combinations")
print(f"Estimated time: ~{total_combos * 1.5:.0f} minutes (assuming ~90s per config)\n")

# Confirm before proceeding
proceed = input("Proceed with grid search? (yes/no): ")
if proceed.lower() != 'yes':
    print("Grid search cancelled.")
else:
    print("\nStarting grid search...")

Grid search will test 72 combinations
Estimated time: ~108 minutes (assuming ~90s per config)



Proceed with grid search? (yes/no):  yes



Starting grid search...


In [45]:
# Run grid search
if proceed.lower() == 'yes':
    grid_results, best_config = tuner.grid_search(
        documents,
        embeddings,
        param_grid,
        max_combinations=None
    )

    print("\n✓ Grid search complete!")
    print(f"\nTop 5 Configurations:")
    print(grid_results.head(5)[[
        'hdbscan.min_cluster_size',
        'umap.n_neighbors',
        'umap.n_components',
        'hdbscan.min_samples',
        'n_topics',
        'coherence',
        'diversity',
        'composite_score'
    ]].to_string(index=False))

2026-01-01 18:08:29,457 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-01 18:08:29,765 - BERTopic - Dimensionality - Completed ✓
2026-01-01 18:08:29,766 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 18:08:29,782 - BERTopic - Cluster - Completed ✓
2026-01-01 18:08:29,785 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 18:08:30,853 - BERTopic - Representation - Completed ✓
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid


✓ Grid search complete!

Top 5 Configurations:
 hdbscan.min_cluster_size  umap.n_neighbors  umap.n_components  hdbscan.min_samples  n_topics  coherence  diversity  composite_score
                        7                10                 10                    3        10   0.576740       0.78         0.710733
                        3                15                  8                    5         2   0.524621       0.95         0.703423
                        4                15                  8                    5         2   0.524621       0.95         0.703423
                        7                15                  4                    3        10   0.590173       0.73         0.695752
                        4                10                  6                    5        10   0.532956       0.79         0.695415


In [47]:
# Save grid search results
if proceed.lower() == 'yes':
    tuner.save_results(
        grid_results,
        OUTPUT_DIR,
        best_config,
        experiment_name=f"grid_search_era{ERA}"
    )

    print(f"\nResults saved to {OUTPUT_DIR}")


Results saved to /kaggle/working/tuning_results


In [31]:
# #FOR ERA 2 - "best" results give only 2 topics so we manually take the next best option with 10 topics
# best_config.umap.n_neighbors = 20
# best_config.umap.n_components = 9
# best_config.hdbscan.min_cluster_size = 5
# best_config.hdbscan.min_samples =5

# # Fix grid_results DataFrame - reorder so the 10-topic config is first
# mask = (
#     (grid_results['umap.n_neighbors'] == 20) &
#     (grid_results['umap.n_components'] == 9) &
#     (grid_results['hdbscan.min_cluster_size'] == 5) &
#     (grid_results['hdbscan.min_samples'] == 5)
# )

# # Move the selected row to the top
# selected_row = grid_results[mask]
# other_rows = grid_results[~mask]
# grid_results = pd.concat([selected_row, other_rows]).reset_index(drop=True)


## 6. Visualize Best Configuration

In [48]:
# Fit model with best config
if proceed.lower() == 'yes':
    # Clear GPU cache and delete embedding model (we have pre-computed embeddings)
    import torch
    if torch.cuda.is_available():
        # Delete the embedding model to free GPU memory
        if 'embedding_model' in globals():
            del embedding_model
        torch.cuda.empty_cache()
        print("🧹 Cleared GPU cache and removed embedding model")

    logger.info("Fitting model with best configuration...")

    # Create modeler but we'll manually set embedding_model=None since we have embeddings
    from bertopic import BERTopic
    from umap import UMAP
    from hdbscan import HDBSCAN
    from sklearn.feature_extraction.text import CountVectorizer

    # Build components without embedding model
    umap_model = UMAP(
        n_neighbors=best_config.umap.n_neighbors,
        n_components=best_config.umap.n_components,
        min_dist=best_config.umap.min_dist,
        random_state=best_config.umap.random_state
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=best_config.hdbscan.min_cluster_size,
        min_samples=best_config.hdbscan.min_samples,
        gen_min_span_tree=best_config.hdbscan.gen_min_span_tree,
        prediction_data=best_config.hdbscan.prediction_data
    )

    vectorizer_model = CountVectorizer(
        ngram_range=best_config.vectorizer.ngram_range,
        stop_words=None
    )

    # Create BERTopic WITHOUT embedding model (we have pre-computed embeddings)
    best_model = BERTopic(
        embedding_model=None,  # Don't load embedding model - saves GPU memory!
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=best_config.bertopic.top_n_words,
        language=best_config.bertopic.language,
        calculate_probabilities=best_config.bertopic.calculate_probabilities,
        verbose=best_config.bertopic.verbose
    )

    # Fit with pre-computed embeddings
    best_topics, best_probs = best_model.fit_transform(documents, embeddings=embeddings)

    print("✓ Best model fitted")
    print(f"Number of topics: {len(set(best_topics)) - (1 if -1 in best_topics else 0)}")
    
    # Evaluate best model with detailed metrics
    best_metrics = tuner.evaluate_model(
        best_model,
        documents,
        embeddings,
        best_topics,
        best_probs
    )
    best_metrics['composite_score'] = tuner.calculate_composite_score(best_metrics)
    
    # Display detailed results
    print("\n" + "="*60)
    print("BEST CONFIGURATION RESULTS")
    print("="*60)
    for key, value in best_metrics.items():
        if isinstance(value, float):
            print(f"{key:25s}: {value}")
        else:
            print(f"{key:25s}: {value}")
    print("="*60)

2026-01-01 18:20:06,893 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


🧹 Cleared GPU cache and removed embedding model


2026-01-01 18:20:07,293 - BERTopic - Dimensionality - Completed ✓
2026-01-01 18:20:07,294 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-01 18:20:07,308 - BERTopic - Cluster - Completed ✓
2026-01-01 18:20:07,310 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-01 18:20:08,416 - BERTopic - Representation - Completed ✓


✓ Best model fitted
Number of topics: 10


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



BEST CONFIGURATION RESULTS
coherence                : 0.5767399484301506
diversity                : 0.78
n_topics                 : 10
outlier_pct              : 0.12121212121212122
avg_docs_per_topic       : 17.4
silhouette               : 0.06438157707452774
min_topic_size           : 7
max_topic_size           : 56
median_topic_size        : 13
composite_score          : 0.7107332411615556


In [49]:
# UMAP visualization
if proceed.lower() == 'yes':
    best_umap = visualizer.plot_umap_2d(
        embeddings,
        best_topics,
        documents,
        metadata,
        title=f"Best Configuration - Era{ERA} Topics",
        save_name=f"best_config_era{ERA}_umap.html"
    )
    print(f"UMAP plot saved to: {OUTPUT_DIR / f'best_config_era{ERA}_umap.html'}")

UMAP plot saved to: /kaggle/working/tuning_results/best_config_era2_umap.html


In [50]:
# Topic distribution
if proceed.lower() == 'yes':
    best_dist = visualizer.plot_topic_distribution(
        best_topics,
        title="Best Configuration - Documents per Topic",
        save_name=f"best_config_era{ERA}_distribution.html"
    )
    print(f"Distribution plot saved to: {OUTPUT_DIR / f'best_config_era{ERA}_distribution.html'}")

Distribution plot saved to: /kaggle/working/tuning_results/best_config_era2_distribution.html


In [51]:
# Configuration comparison radar chart
if proceed.lower() == 'yes':
    comparison = visualizer.plot_configuration_comparison(
        grid_results,
        top_n=5,
        save_name=f"top5_comparison_era{ERA}.html"
    )
    print(f"Comparison plot saved to: {OUTPUT_DIR / f'top5_comparison_era{ERA}.html'}")

Comparison plot saved to: /kaggle/working/tuning_results/top5_comparison_era2.html


In [52]:
# Parameter heatmap (pick two most important parameters)
if proceed.lower() == 'yes':
    heatmap = visualizer.plot_parameter_heatmap(
        grid_results,
        'hdbscan.min_cluster_size',
        'umap.n_neighbors',
        metric='composite_score',
        save_name=f"heatmap_era{ERA}.html"
    )
    print(f"Heatmap saved to: {OUTPUT_DIR / f'heatmap_era{ERA}.html'}")

Heatmap saved to: /kaggle/working/tuning_results/heatmap_era2.html


## 7. Export Best Configuration

In [53]:
# Display best configuration
if proceed.lower() == 'yes':
    print("\n" + "="*60)
    print("BEST CONFIGURATION")
    print("="*60)
    print(f"\nUMAP:")
    print(f"  n_neighbors: {best_config.umap.n_neighbors}")
    print(f"  n_components: {best_config.umap.n_components}")
    print(f"  min_dist: {best_config.umap.min_dist}")
    print(f"\nHDBSCAN:")
    print(f"  min_cluster_size: {best_config.hdbscan.min_cluster_size}")
    print(f"  min_samples: {best_config.hdbscan.min_samples}")
    print(f"\nVectorizer:")
    print(f"  ngram_range: {best_config.vectorizer.ngram_range}")
    print("="*60)


BEST CONFIGURATION

UMAP:
  n_neighbors: 10
  n_components: 10
  min_dist: 0.05

HDBSCAN:
  min_cluster_size: 7
  min_samples: 3

Vectorizer:
  ngram_range: (1, 2)


In [54]:
# Save optimal config for this era
if proceed.lower() == 'yes':
    # Save to writable output directory (not read-only input!)
    optimal_config_path = OUTPUT_DIR / f"era{ERA}_optimal_config.json"
    save_config(best_config, optimal_config_path)
    print(f"\n✓ Saved optimal configuration to {optimal_config_path}")
    print(f"\nYou can download this file and use it to update your config files.")
    print(f"In Kaggle, files are saved to /kaggle/working/ which you can download from the Output tab.")


✓ Saved optimal configuration to /kaggle/working/tuning_results/era2_optimal_config.json

You can download this file and use it to update your config files.
In Kaggle, files are saved to /kaggle/working/ which you can download from the Output tab.


## 8. Compare Baseline vs Best

In [55]:
# Side-by-side comparison
if proceed.lower() == 'yes':
    comparison_df = pd.DataFrame({
        'Metric': ['Coherence', 'Diversity', 'N Topics', 'Outliers (%)', 'Silhouette', 'Composite Score'],
        'Baseline': [
            f"{baseline_metrics['coherence']:.3f}",
            f"{baseline_metrics['diversity']:.3f}",
            baseline_metrics['n_topics'],
            f"{baseline_metrics['outlier_pct']:.1%}",
            f"{baseline_metrics['silhouette']:.3f}",
            f"{baseline_metrics['composite_score']:.3f}"
        ],
        'Best': [
            f"{grid_results.iloc[0]['coherence']:.3f}",
            f"{grid_results.iloc[0]['diversity']:.3f}",
            int(grid_results.iloc[0]['n_topics']),
            f"{grid_results.iloc[0]['outlier_pct']:.1%}",
            f"{grid_results.iloc[0]['silhouette']:.3f}",
            f"{grid_results.iloc[0]['composite_score']:.3f}"
        ]
    })

    print("\n" + "="*60)
    print("BASELINE vs BEST CONFIGURATION")
    print("="*60)
    print(comparison_df.to_string(index=False))
    print("="*60)


BASELINE vs BEST CONFIGURATION
         Metric Baseline  Best
      Coherence    0.543 0.577
      Diversity    0.724 0.780
       N Topics       33    10
   Outliers (%)    24.7% 12.1%
     Silhouette    0.140 0.064
Composite Score    0.599 0.711
